# Memory and Conversation State

Agents and teams accumulate **conversation history** across turns. AutoGen 0.4+ exposes this through **`model_context`** on agents and **`save_state` / `load_state`** on agents and teams — the AgentChat answer to persistent sessions.

This notebook covers **team history**, **state snapshots**, **thread persistence**, **`UnboundedChatCompletionContext` vs `BufferedChatCompletionContext`**, a **multi-turn user loop**, comparison to **LangGraph/08** checkpointing, and a **`user_id` session** pattern.

**Prerequisites:** **08. GroupChat and Multi-Agent Teams**, **09. Speaker Selection**, **11. Sequential and Hierarchical Workflows**.

**Cross-references:** **LangGraph/08** (checkpointing), **13. Termination Conditions**, **16. Production Patterns**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json
from pathlib import Path

try:
    import autogen_agentchat
    print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "installed"))
except ImportError:
    print("pip install autogen-agentchat autogen-ext[openai]")

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.ui import Console
from autogen_core.model_context import BufferedChatCompletionContext, UnboundedChatCompletionContext
from autogen_ext.models.openai import OpenAIChatCompletionClient

In [ ]:
DOC_CHUNKS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files."},
]


def search_docs(query: str) -> str:
    """Keyword search over Notes API documentation chunks."""
    tokens = set(query.lower().split())
    hits = []
    for chunk in DOC_CHUNKS:
        if tokens & set(chunk["text"].lower().split()):
            hits.append(f"[{chunk['id']}] {chunk['text']}")
    return "\n".join(hits) if hits else "No matching chunks."


def describe_endpoint(method: str, path: str) -> str:
    """Describe a Notes API HTTP endpoint (teaching stub)."""
    catalog = {
        ("GET", "/notes"): "List all notes for the authenticated user.",
        ("POST", "/notes"): "Create a note; body requires title and content.",
        ("PUT", "/notes/{id}"): "Update an existing note by id.",
        ("DELETE", "/notes/{id}"): "Delete a note by id.",
    }
    return catalog.get((method.upper(), path), f"Unknown endpoint {method} {path}")

In [ ]:
def make_model_client():
    return OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

model_client = make_model_client()

---

## 1. Where State Lives in AgentChat

| Layer | What is stored | API |
|-------|----------------|-----|
| **Team thread** | Broadcast messages for current task | `TaskResult.messages`, `team.reset()` |
| **Agent `model_context`** | LLM-facing history (virtual view) | `UnboundedChatCompletionContext` / `BufferedChatCompletionContext` |
| **Serialized snapshot** | Team + all participant states | `await team.save_state()` / `load_state` |

```text
team.run → messages appended → each agent updates model_context → optional save_state → JSON/file/DB
```

---

## 2. Notes Assistant — Default (Unbounded) Context

In [ ]:
notes_assistant = AssistantAgent(
    "notes_assistant",
    description="Answers Questions about the Notes API.",
    model_client=model_client,
    tools=[search_docs],
    system_message="Use search_docs. Cite [c#]. Be concise.",
    # Default model_context is UnboundedChatCompletionContext — full history to the model.
)

solo_team = RoundRobinGroupChat(
    [notes_assistant],
    termination_condition=MaxMessageTermination(max_messages=8),
)

---

## 3. `UnboundedChatCompletionContext` vs `BufferedChatCompletionContext`

| Context | Behavior | Use when |
|---------|----------|----------|
| **`UnboundedChatCompletionContext`** | All messages sent to model | Short chats, debugging |
| **`BufferedChatCompletionContext(buffer_size=N)`** | Only last **N** messages to model | Long sessions, token limits |

**Important:** `save_state` still persists the **full** underlying history even when the buffer truncates what the model sees.

In [ ]:
buffered_assistant = AssistantAgent(
    "notes_assistant_buffered",
    description="Notes API helper with trimmed context window.",
    model_client=model_client,
    tools=[search_docs],
    system_message="Use search_docs. Cite [c#].",
    model_context=BufferedChatCompletionContext(buffer_size=6),
)

print("buffer_size=6 — model sees last 6 messages only")

---

## 4. Multi-Turn on the Same Team (No Reset)

Second `run_stream` **continues** the conversation context unless you `reset()`:

In [ ]:
async def two_turn_demo():
    await solo_team.reset()
    await Console(solo_team.run_stream(task="What is the Notes API built with?"))
    await Console(solo_team.run_stream(task="What did I just ask you about?"))


# await two_turn_demo()
print("Second turn should recall prior question")

---

## 5. `save_state` and `load_state` on a Team

Captures **team state** and **all participant agent states** recursively:

In [ ]:
async def save_team_snapshot(team, path: str = "notes_team_state.json") -> dict:
    state = await team.save_state()
    Path(path).write_text(json.dumps(state, indent=2), encoding="utf-8")
    print(f"Saved team state keys: {list(state.keys())}")
    return state


async def load_team_snapshot(team, path: str = "notes_team_state.json") -> None:
    state = json.loads(Path(path).read_text(encoding="utf-8"))
    await team.load_state(state)
    print("Team state loaded")


print("save_team_snapshot / load_team_snapshot helpers defined")

---

## 6. Persist → New Team Instance → Resume

Production services often **reconstruct** the team from config, then `load_state`:

In [ ]:
async def persist_and_resume_demo():
    await solo_team.reset()
    await solo_team.run(task="Search docs for JWT authentication.")
    await save_team_snapshot(solo_team, "notes_team_state.json")

    # Fresh team instance (e.g., after worker restart)
    resumed_assistant = AssistantAgent(
        "notes_assistant",
        description="Answers Questions about the Notes API.",
        model_client=model_client,
        tools=[search_docs],
        system_message="Use search_docs. Cite [c#]. Be concise.",
    )
    resumed_team = RoundRobinGroupChat(
        [resumed_assistant],
        termination_condition=MaxMessageTermination(max_messages=8),
    )
    await load_team_snapshot(resumed_team, "notes_team_state.json")
    result = await resumed_team.run(task="Summarize what we learned about auth.")
    return result


# result = await persist_and_resume_demo()
print("Pattern: construct team → load_state → continue run")

**Caveat:** `save_state` / `load_state` snapshot runtime state — you must rebuild agents with the **same names, tools, and config** when resuming.

---

## 7. Agent-Level `save_state` (LLM Context Only)

In [ ]:
async def agent_context_snapshot():
    agent = AssistantAgent(
        "ctx_demo",
        model_client=model_client,
        system_message="You remember prior turns.",
    )
    team = RoundRobinGroupChat([agent], termination_condition=MaxMessageTermination(max_messages=4))
    await team.run(task="Remember code name ALPHA.")
    agent_state = await agent.save_state()
    print("agent state keys:", list(agent_state.keys()))
    return agent_state


# state = await agent_context_snapshot()
print("AssistantAgent state is primarily llm_context")

---

## 8. Multi-Turn User Loop (Same Team)

In [ ]:
async def notes_chat_loop(team, turns: list[str]):
    for i, user_text in enumerate(turns, 1):
        print(f"--- turn {i} ---")
        result = await team.run(task=user_text)
        last = result.messages[-1]
        reply = last.to_text() if hasattr(last, "to_text") else str(last.content)
        print("assistant:", reply[:200])


# await solo_team.reset()
# await notes_chat_loop(solo_team, ["JWT header?", "Now Alembic autogenerate?", "What were my topics?"])
print("notes_chat_loop ready")

---

## 9. Session per `user_id` Pattern

Map each user to an in-memory or on-disk state blob:

In [ ]:
class NotesSessionStore:
    def __init__(self):
        self._cache: dict[str, dict] = {}

    async def save(self, user_id: str, team) -> None:
        self._cache[user_id] = await team.save_state()

    async def load(self, user_id: str, team) -> bool:
        blob = self._cache.get(user_id)
        if not blob:
            return False
        await team.load_state(blob)
        return True


def make_team_for_user():
    agent = AssistantAgent(
        "notes_assistant",
        model_client=model_client,
        tools=[search_docs],
        system_message="Notes API assistant. Cite [c#].",
    )
    return RoundRobinGroupChat([agent], termination_condition=MaxMessageTermination(max_messages=10))


store = NotesSessionStore()
print("NotesSessionStore: user_id → team state dict")

In [ ]:
async def handle_user_message(user_id: str, message: str) -> str:
    team = make_team_for_user()
    loaded = await store.load(user_id, team)
    if not loaded:
        await team.reset()
    result = await team.run(task=message)
    await store.save(user_id, team)
    last = result.messages[-1]
    return last.to_text() if hasattr(last, "to_text") else str(last.content)


# r = await handle_user_message("user-42", "Describe GET /notes")
print("Production: back store with Redis/Postgres instead of dict")

---

## 10. Compare LangGraph Checkpointing (**LangGraph/08**)

| Feature | AutoGen `save_state` | LangGraph checkpointer |
|---------|---------------------|------------------------|
| Key | Your `user_id` / file path | `configurable.thread_id` |
| Granularity | Team + agents snapshot | Graph state per super-step |
| Resume API | `load_state` + `run` | `invoke` with same `thread_id` |
| Time travel | Manual snapshot management | `get_state_history` |
| Default store | You provide (dict, file, DB) | `MemorySaver`, `SqliteSaver`, `PostgresSaver` |

```text
AutoGen:   save_state() → JSON → load_state() → run(task)
LangGraph: thread_id=A → checkpoint after each node → append message
```

Use **LangGraph** when graph topology + HITL interrupts matter; use **AutoGen state** for chat-team sessions.

---

## 11. Thread Persistence to Disk (File-Backed Sessions)

In [ ]:
STATE_DIR = Path("notes_sessions")
STATE_DIR.mkdir(exist_ok=True)


async def save_session_to_disk(user_id: str, team) -> None:
    state = await team.save_state()
    (STATE_DIR / f"{user_id}.json").write_text(json.dumps(state), encoding="utf-8")


async def load_session_from_disk(user_id: str, team) -> bool:
    path = STATE_DIR / f"{user_id}.json"
    if not path.exists():
        return False
    await team.load_state(json.loads(path.read_text(encoding="utf-8")))
    return True


print("File-backed sessions in", STATE_DIR)

---

## 12. `reset()` vs `load_state`

| Call | Effect |
|------|--------|
| `await team.reset()` | Empty thread — new conversation |
| `await team.load_state(blob)` | Restore prior snapshot — continue conversation |
| New `run` without reset | Append to current in-memory thread |

Always **`reset` OR `load_state`** before starting a unrelated conversation on a reused team object.

---

## 13. Inspect Conversation History

In [ ]:
async def print_team_history(team, task: str):
    result = await team.run(task=task)
    for i, m in enumerate(result.messages):
        src = getattr(m, "source", "?")
        text = m.to_text() if hasattr(m, "to_text") else str(getattr(m, "content", m))
        print(f"{i:02d} [{src}] {text[:80]}...")
    return result


print("print_team_history(team, task)")

---

## 14. Buffered Context Demo — Long Session Behavior

In [ ]:
async def buffered_long_session():
    team = RoundRobinGroupChat(
        [buffered_assistant],
        termination_condition=MaxMessageTermination(max_messages=12),
    )
    await team.reset()
    prompts = [f"Remember token-{i}" for i in range(5)]
    prompts.append("What was token-0?")
    for p in prompts:
        await team.run(task=p)
    # With buffer_size=6, earliest tokens may be invisible to the model — expect possible loss.
    print("Buffered context may forget early turns — by design")


# await buffered_long_session()
print("Use Unbounded when full recall matters; Buffered for token caps")

---

## 15. Termination State

Termination conditions can also expose `save_state` / `load_state` (e.g., message counts). Pair team snapshots with termination reset when starting fresh sessions (**13**).

In [ ]:
term = MaxMessageTermination(max_messages=10)
# term_state = await term.save_state()
print("Reset termination when loading old team state into a new session policy")

---

## 16. `build_notes_memory_team()` Factory

In [ ]:
def build_notes_memory_team(*, buffer_size: int | None = None):
    kwargs = {}
    if buffer_size is not None:
        kwargs["model_context"] = BufferedChatCompletionContext(buffer_size=buffer_size)
    agent = AssistantAgent(
        "notes_assistant",
        description="Notes API assistant with configurable memory.",
        model_client=model_client,
        tools=[search_docs],
        system_message="Use search_docs. Cite [c#].",
        **kwargs,
    )
    return RoundRobinGroupChat([agent], termination_condition=MaxMessageTermination(max_messages=12))


unbounded_team = build_notes_memory_team()
bounded_team = build_notes_memory_team(buffer_size=8)
print("unbounded vs buffer_size=8 teams ready")

---

## 17. Async REPL with Persistence

In [ ]:
async def notes_repl(user_id: str = "demo-user"):
    team = build_notes_memory_team()
    store = NotesSessionStore()
    await store.load(user_id, team)
    queries = ["JWT auth?", "pytest fixtures location?", "exit"]
    for q in queries:
        if q.strip().lower() == "exit":
            break
        reply = await handle_user_message(user_id, q)
        print(">", q)
        print(reply[:300])


# await notes_repl()
print("Wire into FastAPI websocket in 16. Production Patterns")

---

## 18. Summary

| Need | Tool |
|------|------|
| Continue same session | Reuse team or `load_state` |
| Trim model context | `BufferedChatCompletionContext` |
| Full history to model | `UnboundedChatCompletionContext` (default) |
| Export session | `await team.save_state()` → JSON/DB |
| Per-user isolation | `user_id` → state blob (**§9**) |
| Graph-grade checkpointing | **LangGraph/08** `thread_id` + checkpointer |

**Next:** **13. Termination Conditions and Guardrails**, **15. Observability**, **16. Production Patterns**.